# 05 · Silver: vuelos de Brasil y EE. UU. en una sola tabla

**Propósito:** limpiar los vuelos de las dos fuentes y llevarlos a **un mismo esquema**, con todas las horas en
**UTC**, para que el resto del pipeline (clima, features y modelo) los trate igual.

**Entrada:** `bronze/anac_vra` (Brasil), `bronze/bts` (EE. UU.) y `bronze/ref/aeropuertos` (zona horaria de cada
aeropuerto de EE. UU.).

**Salida:** `silver/vuelos/pais=XX/year=AAAA/month=M/` en Parquet comprimido con Snappy, y el conteo de cada regla de
limpieza en `docs/evidencias/silver/conteos_vuelos.csv`.

**Requisito:** notebooks 01, 02 y 03.

Si se vuelve a ejecutar, cada mes reemplaza solo su propia partición: no se duplican datos.

In [1]:
import os
from pathlib import Path

import yaml

# Carpeta del repositorio (este notebook está en notebooks/)
RAIZ = Path.cwd().parent
with open(RAIZ / "config" / "pipeline.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
tablas = cfg["lake"]["tablas"]

# En la EC2 la variable LAKE_BUCKET trae el nombre del bucket; en una laptop se trabaja sobre data/lake
BUCKET = os.environ.get("LAKE_BUCKET", "").strip()
RAIZ_LOCAL = RAIZ / cfg["lake"]["raiz_local"]
BASE = f"s3a://{BUCKET}" if BUCKET else RAIZ_LOCAL.resolve().as_uri()


def ruta(tabla: str) -> str:
    """Ruta completa de una tabla declarada en config/pipeline.yaml."""
    return f"{BASE}/{tablas[tabla]}"


def meses(desde: str, hasta: str) -> list[tuple[int, int]]:
    """Pares (año, mes) entre dos fechas AAAA-MM, ambas incluidas."""
    anio, mes = map(int, desde.split("-"))
    anio_fin, mes_fin = map(int, hasta.split("-"))
    salida = []
    while (anio, mes) <= (anio_fin, mes_fin):
        salida.append((anio, mes))
        anio, mes = (anio + 1, 1) if mes == 12 else (anio, mes + 1)
    return salida


DESDE, HASTA = cfg["periodo"]["inicio"], cfg["periodo"]["fin"]
print("Destino:", f"s3://{BUCKET}" if BUCKET else "data/lake (local)")
print("Periodo:", DESDE, "a", HASTA)

Destino: s3://latam-bigdata-lake-728fd64c
Periodo: 2023-01 a 2025-12


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
import pandas as pd

# En la EC2 Spark toma la configuración del clúster (spark-defaults.conf); en una laptop corre en modo local
constructor = SparkSession.builder.appName("latam-bigdata-silver-vuelos")
if not BUCKET:
    constructor = (
        constructor.master("local[*]")
        .config("spark.sql.shuffle.partitions", str(cfg["spark"]["particiones_shuffle_local"]))
        .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "3g"))
    )
spark = (
    constructor.config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")


def en_periodo(df, col_anio="year", col_mes="month"):
    """Deja solo las filas cuyo año y mes están dentro del periodo de config/pipeline.yaml."""
    ym = F.format_string("%04d-%02d", F.col(col_anio), F.col(col_mes))
    return df.where((ym >= DESDE) & (ym <= HASTA))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/25 07:50:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/25 07:50:40 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


26/11/25 07:50:43 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


26/11/25 07:50:45 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to logs/spark-events/app-20261125075042-0002.inprogress. This is unsupported


In [3]:
EVID = RAIZ / "docs" / "evidencias" / "silver"
EVID.mkdir(parents=True, exist_ok=True)

LATAM = list(cfg["latam"]["pasajeros"])
LATAM_CARGA = list(cfg["latam"]["carga"])
UMBRAL_AMBAR = cfg["umbrales_demora_min"]["ambar"]
UMBRAL_ROJO = cfg["umbrales_demora_min"]["rojo"]
DEMORA_MIN, DEMORA_MAX = cfg["calidad"]["demora_min"], cfg["calidad"]["demora_max"]
ZONA_BR = cfg["fuentes"]["vra"]["zona_horaria"]


def texto(c):
    """Texto sin espacios; vacío o la palabra 'null' pasan a nulo."""
    v = F.trim(F.col(c))
    return F.when((v == "") | (F.lower(v) == "null"), None).otherwise(v)


def sumar_minutos(ts, minutos):
    """Suma minutos (pueden ser negativos o nulos) a un timestamp."""
    return F.timestamp_seconds(F.unix_timestamp(ts) + (minutos * 60).cast("long"))

## 1. Brasil (ANAC VRA)

- Las horas vienen en hora de Brasília. Desde octubre de 2025 traen fracciones de segundo, así que se recortan a
  `AAAA-MM-DD HH:MM:SS` antes de convertirlas.
- La conversión a UTC usa la zona `America/Sao_Paulo`: es la hora en que ANAC publica, también los vuelos
  internacionales.
- La demora es la diferencia entre la salida real y la programada. VRA no informa desvíos.
- La fecha y la hora **local** de salida no son las de Brasília sino las del aeropuerto de origen: se calculan
  en la sección 2b, cuando ya se tiene la ficha de aeropuertos.

In [4]:
# Las 12 columnas de VRA, en el orden del archivo. Se leen como texto y se tipan después.
COLUMNAS_VRA = [
    "empresa", "numero_voo", "codigo_di", "tipo_linha", "origem", "destino",
    "partida_prevista", "partida_real", "chegada_prevista", "chegada_real", "situacao", "justificativa",
]
esquema_vra = T.StructType([T.StructField(c, T.StringType()) for c in COLUMNAS_VRA])

vra_crudo = (
    spark.read.option("sep", ";").option("quote", '"').schema(esquema_vra)
    .csv(ruta("bronze_vra"))  # year y month salen de las carpetas year=/month=
)
# Fuera la línea "Atualizado em ..." y la fila de encabezado de cada archivo
vra_crudo = en_periodo(
    vra_crudo.where(~F.col("empresa").contains("Atualizado") & (F.col("empresa") != "ICAO Empresa Aérea"))
)

In [5]:
def hora_vra(c):
    return F.to_timestamp(F.substring(texto(c), 1, 19), "yyyy-MM-dd HH:mm:ss")


br = vra_crudo.select(
    F.lit("BR").alias("pais"),
    texto("empresa").alias("aerolinea"),
    texto("numero_voo").alias("numero_vuelo"),
    texto("origem").alias("origen"),
    texto("destino").alias("destino"),
    hora_vra("partida_prevista").alias("salida_prog_local"),
    hora_vra("partida_real").alias("salida_real_local"),
    hora_vra("chegada_prevista").alias("llegada_prog_local"),
    hora_vra("chegada_real").alias("llegada_real_local"),
    F.coalesce(texto("situacao") == "CANCELADO", F.lit(False)).alias("cancelado"),
)
br = br.select(
    "pais", "aerolinea", "numero_vuelo", "origen", "destino", "cancelado",
    F.lit(False).alias("desviado"),
    F.lit(True).alias("tiene_zona_horaria"),
    "salida_prog_local",
    F.to_utc_timestamp("salida_prog_local", ZONA_BR).alias("salida_prog_utc"),
    F.to_utc_timestamp("salida_real_local", ZONA_BR).alias("salida_real_utc"),
    F.to_utc_timestamp("llegada_prog_local", ZONA_BR).alias("llegada_prog_utc"),
    F.to_utc_timestamp("llegada_real_local", ZONA_BR).alias("llegada_real_utc"),
)
br = br.withColumn(
    "demora_salida_min", (F.unix_timestamp("salida_real_utc") - F.unix_timestamp("salida_prog_utc")) / 60
).withColumn(
    "demora_llegada_min", (F.unix_timestamp("llegada_real_utc") - F.unix_timestamp("llegada_prog_utc")) / 60
)

## 2. EE. UU. (BTS)

- Se toman solo las columnas necesarias.
- Las horas `hhmm` están en hora local de cada aeropuerto: se arman sobre la fecha del vuelo (`2400` cae en la
  medianoche del día siguiente) y se convierten a UTC con la zona horaria IANA del aeropuerto, que respeta el
  horario de verano.
- La salida y la llegada reales se calculan como la hora programada más la demora informada por BTS: así no hay que
  adivinar si el vuelo cruzó la medianoche.
- Si la llegada programada en UTC queda antes que la salida, el vuelo llega al día siguiente.
- Los vuelos cuyo aeropuerto no tiene zona horaria en la ficha no se pueden convertir a UTC: se cuentan y se excluyen.

In [6]:
# BTS trae unas 110 columnas. El esquema se arma con los nombres del propio encabezado y todas se leen
# como texto (sin inferir tipos); después se toman y se tipan solo las que se usan.
encabezado = (
    spark.read.text(ruta("bronze_bts")).where(F.col("value").startswith('"Year"')).first()["value"]
)
nombres_bts = [n.strip('"') or f"_vacia_{i}" for i, n in enumerate(encabezado.split(","))]
# Year y Month chocan con las columnas year/month que Spark toma de las carpetas: se renombran
nombres_bts = [f"{n}_bts" if n.lower() in ("year", "month") else n for n in nombres_bts]
esquema_bts = T.StructType([T.StructField(n, T.StringType()) for n in nombres_bts])

# El encabezado de cada archivo se descarta como fila (así Spark no compara nombres con el esquema)
bts_crudo = en_periodo(
    spark.read.option("header", False).schema(esquema_bts).csv(ruta("bronze_bts"))
    .where(F.col("FlightDate") != "FlightDate")
)

In [7]:
esquema_aeropuertos = T.StructType([
    T.StructField("aeropuerto", T.StringType()), T.StructField("estacion", T.StringType()),
    T.StructField("nombre", T.StringType()), T.StructField("pais", T.StringType()),
    T.StructField("latitud", T.DoubleType()), T.StructField("longitud", T.DoubleType()),
    T.StructField("zona_horaria", T.StringType()), T.StructField("tiene_metar", T.BooleanType()),
])
zonas = (
    spark.read.option("header", True).schema(esquema_aeropuertos).csv(ruta("bronze_aeropuertos"))
    .select("aeropuerto", "zona_horaria").where(F.col("zona_horaria").isNotNull())
)


def minutos_hhmm(c):
    v = F.lpad(texto(c), 4, "0")
    return F.substring(v, 1, 2).cast("int") * 60 + F.substring(v, 3, 2).cast("int")


def hora_local_bts(c):
    medianoche = F.unix_timestamp(F.col("FlightDate"), "yyyy-MM-dd")
    return F.timestamp_seconds(medianoche + minutos_hhmm(c) * 60)


us = (
    bts_crudo.select(
        "FlightDate", "Reporting_Airline", "Flight_Number_Reporting_Airline", "Origin", "Dest",
        "CRSDepTime", "CRSArrTime", "DepDelay", "ArrDelay", "Cancelled", "Diverted",
    )
    # La tabla de zonas es chica: se envía completa a cada nodo (broadcast) y el join no mueve los vuelos
    .join(F.broadcast(zonas.withColumnRenamed("aeropuerto", "Origin").withColumnRenamed("zona_horaria", "tz_origen")),
          "Origin", "left")
    .join(F.broadcast(zonas.withColumnRenamed("aeropuerto", "Dest").withColumnRenamed("zona_horaria", "tz_destino")),
          "Dest", "left")
)
us = us.select(
    F.lit("US").alias("pais"),
    texto("Reporting_Airline").alias("aerolinea"),
    texto("Flight_Number_Reporting_Airline").alias("numero_vuelo"),
    texto("Origin").alias("origen"),
    texto("Dest").alias("destino"),
    F.coalesce(F.col("Cancelled").cast("double") == 1, F.lit(False)).alias("cancelado"),
    F.coalesce(F.col("Diverted").cast("double") == 1, F.lit(False)).alias("desviado"),
    (F.col("tz_origen").isNotNull() & F.col("tz_destino").isNotNull()).alias("tiene_zona_horaria"),
    hora_local_bts("CRSDepTime").alias("salida_prog_local"),
    hora_local_bts("CRSArrTime").alias("llegada_prog_local"),
    F.col("DepDelay").cast("double").alias("demora_salida_min"),
    F.col("ArrDelay").cast("double").alias("demora_llegada_min"),
    "tz_origen", "tz_destino",
)
us = us.withColumn("salida_prog_utc", F.to_utc_timestamp("salida_prog_local", F.col("tz_origen")))
us = us.withColumn("llegada_prog_utc", F.to_utc_timestamp("llegada_prog_local", F.col("tz_destino")))
us = us.withColumn(
    "llegada_prog_utc",
    F.when(F.col("llegada_prog_utc") < F.col("salida_prog_utc"), sumar_minutos(F.col("llegada_prog_utc"), F.lit(1440)))
    .otherwise(F.col("llegada_prog_utc")),
)
us = (
    us.withColumn("salida_real_utc", sumar_minutos(F.col("salida_prog_utc"), F.col("demora_salida_min")))
    .withColumn("llegada_real_utc", sumar_minutos(F.col("llegada_prog_utc"), F.col("demora_llegada_min")))
    .drop("llegada_prog_local", "tz_origen", "tz_destino")
)

## 2b. Hora local de los vuelos de Brasil

`hora_local_salida` y `fecha_local` son la hora y la fecha programadas **en el aeropuerto de origen**, igual que en
EE. UU.: la hora del día es una variable del modelo y la congestión se cuenta por día y hora locales. ANAC publica en
hora de Brasília, pero Manaus o Rio Branco van una o dos horas por detrás, así que un vuelo de las 00:30 de Brasília
sale a las 23:30 del día anterior en Manaus.

La salida programada (ya en UTC) se lleva a la zona de cada origen con la ficha de aeropuertos. Los orígenes que no
tienen ficha (aeropuertos donde LATAM no opera) conservan la hora de Brasília.

In [8]:
zonas_br = F.broadcast(
    zonas.withColumnRenamed("aeropuerto", "origen").withColumnRenamed("zona_horaria", "tz_origen")
)
br = (
    br.join(zonas_br, "origen", "left")
    .withColumn(
        "salida_prog_local",
        F.from_utc_timestamp("salida_prog_utc", F.coalesce(F.col("tz_origen"), F.lit(ZONA_BR))),
    )
    .drop("tz_origen")
)

## 3. Un solo esquema

Las dos fuentes se unen por nombre de columna y se agregan los campos comunes:

| Campo | Regla |
|---|---|
| `es_latam` | vuelo de pasajeros del grupo LATAM en Brasil |
| `es_atipico` | demora fuera de [−60, 1440] min: se conserva en silver pero no se usa para entrenar |
| `y_demora15`, `y_demora60` | 1 si salió con 15 o 60 minutos de demora o más (umbrales de config); nulo si fue cancelado o desviado |
| `hora_local_salida`, `fecha_local` | hora y fecha programadas en hora local del origen |
| `id_vuelo` | huella de la clave natural (país, aerolínea, número, origen, salida programada) |

In [9]:
vuelos = br.unionByName(us)
vuelos = (
    vuelos.withColumn("es_latam", (F.col("pais") == "BR") & F.col("aerolinea").isin(LATAM))
    .withColumn("es_latam_carga", (F.col("pais") == "BR") & F.col("aerolinea").isin(LATAM_CARGA))
    .withColumn("hora_local_salida", F.hour("salida_prog_local"))
    .withColumn("fecha_local", F.to_date("salida_prog_local"))
    .withColumn("es_atipico", F.coalesce(
        (F.col("demora_salida_min") < DEMORA_MIN) | (F.col("demora_salida_min") > DEMORA_MAX), F.lit(False)))
)
etiqueta_valida = ~F.col("cancelado") & ~F.col("desviado") & F.col("demora_salida_min").isNotNull()
vuelos = (
    vuelos.withColumn("y_demora15", F.when(etiqueta_valida, (F.col("demora_salida_min") >= UMBRAL_AMBAR).cast("int")))
    .withColumn("y_demora60", F.when(etiqueta_valida, (F.col("demora_salida_min") >= UMBRAL_ROJO).cast("int")))
    .withColumn("year", F.year("fecha_local"))
    .withColumn("month", F.month("fecha_local"))
)
completo = (
    F.col("salida_prog_utc").isNotNull() & F.col("aerolinea").isNotNull() & F.col("numero_vuelo").isNotNull()
    & F.col("origen").isNotNull() & F.col("destino").isNotNull()
)

## 4. Conteo de entrada

Antes de filtrar se cuenta, por país, cuántas filas llegan y cuántas se pierden por cada motivo. Se hace en una sola
pasada sobre los CSV.

In [10]:
ym = F.format_string("%04d-%02d", F.col("year"), F.col("month"))
conteo_entrada = (
    vuelos.groupBy("pais").agg(
        F.count("*").alias("leidas"),
        F.sum((~F.col("tiene_zona_horaria")).cast("int")).alias("sin_zona_horaria"),
        F.sum((F.col("tiene_zona_horaria") & ~completo).cast("int")).alias("sin_datos_clave"),
        F.sum((F.col("tiene_zona_horaria") & completo & ~((ym >= DESDE) & (ym <= HASTA))).cast("int"))
        .alias("fuera_de_periodo"),
    ).toPandas().set_index("pais")
)
conteo_entrada

,leidas,sin_zona_horaria,sin_datos_clave,fuera_de_periodo
pais,,,,
BR,2972195,0,92242,31
US,20928579,181364,1,0


## 5. Deduplicación y escritura

Un mismo vuelo (país, aerolínea, número, origen y salida programada) puede aparecer más de una vez. Se queda la fila
que tiene salida real informada. Antes de escribir se reparte por partición para dejar un archivo por mes y país, en
lugar de cientos de archivos chicos.

In [11]:
CLAVE = ["pais", "aerolinea", "numero_vuelo", "origen", "salida_prog_utc"]
orden = Window.partitionBy(*CLAVE).orderBy(F.col("salida_real_utc").asc_nulls_last(), F.col("llegada_prog_utc"))

silver = (
    vuelos.where(F.col("tiene_zona_horaria") & completo & (ym >= DESDE) & (ym <= HASTA))
    .withColumn("_n", F.row_number().over(orden)).where("_n = 1").drop("_n")
    .withColumn("id_vuelo", F.substring(F.sha2(F.concat_ws(
        "|", "pais", "aerolinea", "numero_vuelo", "origen", F.date_format("salida_prog_utc", "yyyy-MM-dd HH:mm")
    ), 256), 1, 24))
    .select(
        "id_vuelo", "pais", "aerolinea", "es_latam", "es_latam_carga", "numero_vuelo", "origen", "destino",
        "salida_prog_utc", "salida_real_utc", "llegada_prog_utc", "llegada_real_utc",
        "hora_local_salida", "fecha_local", "cancelado", "desviado",
        "demora_salida_min", "demora_llegada_min", "es_atipico", "y_demora15", "y_demora60", "year", "month",
    )
)
(
    silver.repartition("pais", "year", "month")
    .write.mode("overwrite").partitionBy("pais", "year", "month")
    .parquet(ruta("silver_vuelos"))
)
print("silver/vuelos escrito")

silver/vuelos escrito


## 6. Verificación

Se lee lo escrito (Parquet, rápido) y se completan los conteos: duplicados descartados, cancelados, desviados,
atípicos y tasa de demora.

In [12]:
escrito = en_periodo(spark.read.parquet(ruta("silver_vuelos")))
conteo_salida = (
    escrito.groupBy("pais").agg(
        F.count("*").alias("escritas"),
        F.sum(F.col("cancelado").cast("int")).alias("cancelados"),
        F.sum(F.col("desviado").cast("int")).alias("desviados"),
        F.sum(F.col("es_atipico").cast("int")).alias("atipicos"),
        F.count("y_demora15").alias("con_etiqueta"),
        F.round(100 * F.avg("y_demora15"), 2).alias("pct_demora15"),
        F.round(100 * F.avg("y_demora60"), 2).alias("pct_demora60"),
        F.sum(F.col("es_latam").cast("int")).alias("vuelos_latam"),
    ).toPandas().set_index("pais")
)
conteos = conteo_entrada.join(conteo_salida)
conteos["duplicadas"] = (
    conteos["leidas"] - conteos["sin_zona_horaria"] - conteos["sin_datos_clave"] - conteos["fuera_de_periodo"]
    - conteos["escritas"]
)
conteos.reset_index().to_csv(EVID / "conteos_vuelos.csv", index=False)
conteos.T

pais,BR,US
leidas,2972195.00,20928579.00
sin_zona_horaria,0.00,181364.00
sin_datos_clave,92242.00,1.00
fuera_de_periodo,31.00,0.00
escritas,2879784.00,20747214.00
cancelados,119602.00,285616.00
desviados,0.00,52809.00
atipicos,6358.00,1742.00
con_etiqueta,2760182.00,20408789.00
pct_demora15,18.78,20.97


In [13]:
# Control de la conversión a UTC: un mismo vuelo programado a las 08:00 locales en distintas zonas horarias
(
    escrito.where((F.col("pais") == "US") & (F.col("hora_local_salida") == 8))
    .join(F.broadcast(zonas.withColumnRenamed("aeropuerto", "origen")), "origen")
    .groupBy("zona_horaria").agg(F.first("salida_prog_utc").alias("ejemplo_salida_utc"), F.count("*").alias("vuelos"))
    .orderBy(F.desc("vuelos")).limit(6).toPandas()
)

,zona_horaria,ejemplo_salida_utc,vuelos
0,America/New_York,2024-07-05 12:52:00,548482
1,America/Chicago,2024-07-07 13:30:00,366305
2,America/Los_Angeles,2024-07-12 15:24:00,258182
3,America/Denver,2024-07-23 14:25:00,90851
4,America/Phoenix,2024-07-06 15:49:00,52286
5,America/Detroit,2024-07-20 12:50:00,49125


In [14]:
escrito.where(F.col("es_latam")).limit(5).toPandas()

,id_vuelo,aerolinea,es_latam,es_latam_carga,numero_vuelo,origen,destino,salida_prog_utc,salida_real_utc,llegada_prog_utc,...,cancelado,desviado,demora_salida_min,demora_llegada_min,es_atipico,y_demora15,y_demora60,pais,year,month
0,54c0955340b32ed16833533d,ARE,True,False,4903,SKBO,SBGR,2025-01-25 02:20:00,2025-01-25 02:20:00,2025-01-25 08:35:00,...,False,False,0.0,19.0,False,0,0,BR,2025,1
1,c6f1d906dde3a4eba21173a3,ARE,True,False,4904,SBGR,SKBO,2025-01-19 11:00:00,2025-01-19 11:06:00,2025-01-19 17:14:00,...,False,False,6.0,-29.0,False,0,0,BR,2025,1
2,da8e86fcf4a54fb93ab2f5f8,LAN,True,False,0406,SUMU,SBGR,2025-01-12 14:30:00,2025-01-12 14:45:00,2025-01-12 17:05:00,...,False,False,15.0,3.0,False,1,0,BR,2025,1
3,e814e4721a780ce4dd676025,LAN,True,False,0406,SUMU,SBGR,2025-01-23 14:30:00,2025-01-23 14:17:00,2025-01-23 17:05:00,...,False,False,-13.0,-17.0,False,0,0,BR,2025,1
4,7c6e119ecca5afaf6f497955,LAN,True,False,0407,SBGR,SUMU,2025-01-02 18:05:00,2025-01-02 18:03:00,2025-01-02 20:45:00,...,False,False,-2.0,8.0,False,0,0,BR,2025,1


In [15]:
spark.stop()